# __Support Vector Machines__
## _Supervised Learning without Blackboxes_
A tutorial by Vincent Voigtländer (October 2024)

This tutorial introduces Support Vector Machines (SVMs), a supervised learning algorithm for classification tasks. It begins by defining classification problems and illustrating the need for SVMs through a real-world example. The tutorial explains the mathematical foundations of SVMs, covering concepts such as the separating line, canonical form, and margins. Finally, it guides readers through practical implementation using the scikit-learn library.

### Objectives of this tutorial:

- **[Understand classification problems](#first-section)**  
  Learn what classification problems are and how they are used to categorize data into distinct groups.

- **[Explore the mathematics behind support vector machines](#second-section)**  
  Understand the core mathematical concepts of support vector machines, including the separating line, the canonical form, and margins.

- **[Apply support vector machines to data classification](#third-section)**  
  Discover how SVMs are implemented in practice to classify data, with real-world examples.


### Recommended Online Tutorials:
- ⁠In their video [Support Vector Machine (SVM) in 2 minutes](https://youtu.be/_YPScrckx28?feature=shared), the creators of the YouToube channel "Visually Explained" do a great job of illustrating a vivid illustration of the core idea of SVMs. In this tutorial, I also cover the mathematics behind SVMs.
- The paper [Support Vector Machines (SVM)](https://www.mathematik.uni-ulm.de/stochastik/lehre/ss07/seminar_sl/fischer.pdf) by Jasmin Fischer does a great job of explaining the mathematical concepts. However, its length and limited real-world examples can make it a somewhat dense read.
- The [Wikipedia page](https://en.wikipedia.org/wiki/Support_vector_machine) on support vector machines also provides a well-written and detailed deep dive into the topic. Here, however, I focus on the core ideas and concepts of SVMs and provide an interactive environment for hands-on experimentation.

## 1. What is a Classification Problem?
<a class="anchor" id="first-section"></a>

Imagine you are a young botanist and you have been asked to count the number of iris flowers on your grandfather's farm. You know exactly what an Iris flower looks like, but there is a problem: your grandfather has told you that there are **two Iris species** in the area - _Iris setosa_ and _Iris versicolor_ - and he wants you to **count the numbers for each of them separately**.

![_Iris setosa_](resources/IrisSetosa.jpeg)  
_Iris setosa_ (&copy; [Alexey Yakovlev, 2016](https://commons.wikimedia.org/wiki/File:Iris_setosa_(Iridaceae)_(49869832502).jpg))

![_Iris versicolor](resources/IrisVersicolor.jpeg)  
_Iris versicolor_ (&copy; [Nichole Ouellette, 2015](https://commons.wikimedia.org/wiki/File:Iris_versicolor_0007.jpg))



The two species are difficult to distinguish by eye, but fortunately, you found a dataset that lists the **[sepals](https://en.wikipedia.org/wiki/Sepal) length and width** for 50 _Iris setosa_ and 50 _Iris versicolor_ flowers. We can open and look at the dataset using the [pandas](https://pandas.pydata.org) library in Python:

In [ ]:
# import the pandas library. 'pd' is a common shorthand for pandas.
import pandas as pd

# open a .csv-file and read it into a pandas DataFrame
df = pd.read_csv('resources/iris.csv')
# view the DataFrame
df

As it is difficult to make sense of a table with 100 rows of numbers, we will display the data using the [matplotlib](https://matplotlib.org/) library:

In [ ]:
# set backend to interactive widget mode for notebooks
%matplotlib widget
# import the pyplot module of matplotlib with the alias 'plt'
import matplotlib.pyplot as plt

# get subsets of the DataFrame for the different species
setosa = df.loc[df.species == 'setosa']
versicolor = df.loc[df.species == 'versicolor']

# create a new figure and axis
fig1, ax1 = plt.subplots()

# create a scatter plot with both subsets
ax1.scatter(setosa['sepal length (cm)'], setosa['sepal width (cm)'])
ax1.scatter(versicolor['sepal length (cm)'], versicolor['sepal width (cm)'])

# add labels to the axes
ax1.set_xlabel('sepal length (cm)')
ax1.set_ylabel('sepal width (cm)')
# add a title to the plot
ax1.set_title('Sepal length vs. sepal width of Iris setosa and versicolor')
# add a legend to the plot
ax1.legend(['Iris setosa', 'Iris versicolor'], loc='upper right')
ax1.set_aspect('equal')

plt.show()

As you look at the picture, you realize what to do. For each flower, you will measure the **length** and **width** of the sepal and place a corresponding dot on the diagram. 

- If the dot falls within the cluster of **blue points**, the flower is an _Iris setosa_.
- If the dot lies within the **orange cluster**, then your flower is an _Iris versicolor_.


Our task is known as a **Classification** problem:  
> "Classification is a supervised machine learning method where the model tries to predict the correct label of a given input data" ([datacamp.com](https://www.datacamp.com/blog/classification-machine-learning)).  

The labels we aim to predict are the species _Iris setosa_ or _Iris versicolor_. The input data consists of two features: **sepal length** and **sepal width**.


## 2. The Math behind Support Vector Machines <a class="anchor" id="second-section"></a>

Drawing and categorising the dots one by one is a tedious process. That's why we'd like to automate the task. Therefore we come up with a new idea:
We draw a **straight line** in our plot **to separate the two classes**. For each new flower, we can tell which class it is by looking at which side of the line its data point is on.

### The Separating Line

A line $\mathcal{L_0}$ in two dimensions can be defined by the equation 
$$ \mathcal{L_0} \overset{!}{=} \{\boldsymbol x \in \mathbb{R}^2 \mid \boldsymbol x \cdot \boldsymbol w = 0\}. $$
Our line $\mathcal{L_0}$ is expressed as the set of vectors with two real number elements, that satisfy the equation $\boldsymbol x \cdot \boldsymbol w = 0$, where "$\cdot$" denotes the dot product and $\boldsymbol w$ is a vector normal (that means perpendicular) to the line. In other words, the line is the set of all points whose location vector is orthogonal to $\boldsymbol w$. 

Since the null vector $\begin{pmatrix} 0 & 0 \end{pmatrix}^T$ always satisfies our equation, all lines $\mathcal{L_0}$ will go though the origin. To avoid this, we move our line along a vector $\boldsymbol {\bar x}$, which will then always be part of the line:
$$ \mathcal{L} \overset{!}{=} \{\boldsymbol x \in \mathbb{R}^2 \mid (\boldsymbol x - \boldsymbol {\bar x}) \cdot \boldsymbol w = 0\}. $$

The following plot shows our classification problem with the line $\mathcal{L}$. Use the sliders to manipulate the components of $\boldsymbol {\bar x}$ and $\boldsymbol w$ so that the line divides the two classes.

In [1]:
from ipywidgets import interact, FloatSlider
from resources.interactive_plot import plot_iris 

fig2, ax2 = plot_iris(df) # create a plot simular to the one above

xmin, xmax = ax2.get_xlim()
ymin, ymax = ax2.get_ylim()
xbar1 = (xmin + xmax) / 2
xbar2 = (ymin + ymax) / 2
w1 = 1.0
w2 = 0.5
line, point, arrow = None, None, None

def update(xbar1 = xbar1, xbar2 = xbar2, w1 = w1, w2 = w2):
    global line, point, arrow
    if line: line.remove()
    if point: point.remove()
    if arrow: arrow.remove()
    slope = -w1/w2 if w2 != 0 else 1e6
    line = ax2.axline((xbar1, xbar2), slope=slope, color = 'C2')
    point = ax2.scatter(xbar1, xbar2, color='C2')
    arrow = ax2.arrow(xbar1, xbar2, w1, w2, color='C2', head_width=0.1, head_length=0.1)
    fig2.canvas.draw_idle()

interact(
    update, 
    xbar1=FloatSlider(xbar1, min=4.5, max=7, step=0.01, description=r'$\bar x_1$'), 
    xbar2=FloatSlider(xbar2, min=2, max=4.5, step=0.01, description=r'$\bar x_2$'), 
    w1=FloatSlider(w1, min=-1, max=1, step=0.01, description=r'$w_1$'), 
    w2=FloatSlider(w2, min=-1, max=1, step=0.01, description=r'$w_2$'),
);

NameError: name 'df' is not defined

As the dot product is distributive, we can also write
\begin{align*}
\mathcal{L} &= \{\boldsymbol x \in \mathbb{R}^2 \mid (\boldsymbol x - \boldsymbol {\bar x}) \cdot \boldsymbol w = 0\} \\
\mathcal{L} &= \{\boldsymbol x \in \mathbb{R}^2 \mid \boldsymbol x \cdot \boldsymbol w - \boldsymbol {\bar x} \cdot \boldsymbol w = 0\} \\
\mathcal{L} &= \{\boldsymbol x \in \mathbb{R}^2 \mid \boldsymbol x \cdot \boldsymbol w - b = 0\}
\end{align*}
with the offset $b = \boldsymbol {\bar x} \cdot \boldsymbol w$.

It can easily be seen that all non-zero scalar multiples of $\boldsymbol w$ (parallel vectors) yield the same line. For example the line with $\boldsymbol w = \begin{pmatrix} 1 & 1 \end{pmatrix}^T$ equals the line when $\boldsymbol w = \begin{pmatrix} -0.5 & -0.5 \end{pmatrix}^T$. It is therefore a convention to restrict the length of $\boldsymbol w$ to obtain the so-called __canonical form__ of the line.

### Canonical Form

We can illustrate the term $(\boldsymbol x \cdot \boldsymbol w - b)$ from our line equation by plotting its result for each point as a coulor.

In [ ]:
import numpy as np

fig3, ax3 = plot_iris(df) # create a plot simular to the one above

xmin, xmax = ax3.get_xlim()
ymin, ymax = ax3.get_ylim()
xbar1 = 4.7
xbar2 = 2.35
w1 = 0.768
w2 = -0.640
abs_w = 8.223
slope = -w1/w2 if w2 != 0 else 1e6
ax3.axline((xbar1, xbar2), slope=slope, color = 'C2')
XX = np.linspace(xmin, xmax, 100)
YY = np.linspace(ymin, ymax, 100)
XX, YY = np.meshgrid(XX, YY)

def calculate_ZZ(abs_w):
    return (XX - xbar1) * w1 * abs_w + (YY - xbar2) * w2 * abs_w
ZZ = calculate_ZZ(abs_w)

pcm = ax3.pcolormesh(XX, YY, ZZ, zorder=0, shading='auto', cmap='bwr', 
                     vmin=-34, vmax=34)
cs = ax3.contour(XX, YY, ZZ, levels=np.arange(-34, 34, 1), colors='gray', 
                 linewidths=0.5, zorder=0)
cbar = fig3.colorbar(pcm, ax=ax3)
cbar.set_label(r"$(\mathbf{x} \cdot \mathbf{w} - b)$", fontsize=12)

def update(abs_w=abs_w):
    global pcm, cs
    if pcm: pcm.remove()
    if cs: cs.remove()
    ZZ = calculate_ZZ(abs_w)
    pcm = ax3.pcolormesh(XX, YY, ZZ, zorder=0, shading='auto', cmap='bwr', 
                         vmin=-34, vmax=34)
    cs = ax3.contour(XX, YY, ZZ, levels=np.arange(-34, 34, 1), colors='gray', 
                     linewidths=0.5, zorder=0)
    fig3.canvas.draw_idle()

interact(
    update, 
    abs_w=FloatSlider(abs_w, min=0, max=15, step=0.1, description=r'$| \mathbf{w} |$')
);

Use the slider to change the absolute value of $\boldsymbol w$. Try finding the value of $|\boldsymbol w|$ such that the value of $(\boldsymbol x \cdot \boldsymbol w - b)$ equals positive or negative one for the points closest to the line.

The plot shows that the absolute value of the term $(\boldsymbol x \cdot \boldsymbol w - b)$ proportional to the distance between a given point $\boldsymbol x$ and the line. It's sign tells us on which side of the line the point $\boldsymbol x$ is on. Hence, we can use
$$ f(x) \overset{!}{=} \operatorname{sgn}(\boldsymbol x \cdot \boldsymbol w - b) $$
as the decision function to predict the class label $y$ given the input data $\boldsymbol x$.

You can also see from the plot, that the length of $\boldsymbol w$ determines how fast the expression $|\boldsymbol x \cdot \boldsymbol w - b|$ grows as we move $\boldsymbol x$ further away from the line. For the canonical form of $\mathcal{L}$, we pick the length of $\boldsymbol w$ such that the expression $\boldsymbol x \cdot \boldsymbol w - b$ equals $1$ or $-1$ for the data points closest to our line:
$$ \min_{i = 1,\ldots,N} | \boldsymbol x_i \cdot \boldsymbol w - b | = 1. $$
The points $\boldsymbol x$ closest to the line are also called the __support vectors__.

### The Margin

We consider two support vectors $\boldsymbol x_i$ and $\boldsymbol x_j$ which represent the points closest to the line on different sides:
\begin{align*}
    \boldsymbol x_i \cdot \boldsymbol w - b &= +1 \\
    \boldsymbol x_j \cdot \boldsymbol w - b &= -1.
\end{align*}

We subtract both equations
\begin{align*}
    (\boldsymbol x_i -  \boldsymbol x_j) \cdot \boldsymbol w &= 2 \\
    (\boldsymbol x_i -  \boldsymbol x_j) \cdot \frac{\boldsymbol w}{\left \| \boldsymbol w \right \|} &= \frac{2}{\left \| \boldsymbol w \right \|},
\end{align*}
and note that the distance between both points, projected along the direction of $\boldsymbol w$, equals $\frac{2}{\left \| \boldsymbol w \right \|}$. This distance is called the __margin__.

![SVM margin](resources/SVM_margin.png)  
SVM margin (&copy; [Larhmam, 2018](https://commons.wikimedia.org/wiki/File:SVM_margin.png))

### The Optimisation Problem

We want to find the line $\mathcal{L}$ that correctly classifies all points with the largest possible margin.
Visually it becomes clear that the solution in this case only depends on the support vectors - the points closest to the line.

Since we want to maximise the margin $\frac{2}{\left \| \boldsymbol w \right \|}$, we can also minimize $\left \| \boldsymbol w \right \|$.  
By choosing $y \in \{-1, +1\}$ as class labels, the input is correctly classified if 
$$ y_i = f(\boldsymbol x_i) = \operatorname{sgn}(\boldsymbol x_i \cdot \boldsymbol w - b) $$
for all $i \in \{1, \dots, N\}$, where N is the number of inputs. Given the canonical form, we can check
$$ y_i (\boldsymbol x_i \cdot \boldsymbol w - b) \geqq 1 $$
for all $i$, since the sign of $y_i$ and $\boldsymbol x_i \cdot \boldsymbol w - b$ should always match.

Summing up, we need to solve the following:
\begin{align*}
&\operatorname{minimise} && \frac{1}{2}\|\boldsymbol{w}\|^2\\
&\text{subject to} && y_i (\boldsymbol x_i \cdot \boldsymbol w - b) \geqq 1 \quad \forall i \in \{1,\dots,N\}
\end{align*}

A multitude of algorithms can be employed to solve the given optimization problem. The most common solutions include:

- **Transformation into a dual problem**, which is subsequently solved using Lagrange multipliers.
- **Sub-gradient descent**
- **Coordinate descent**

To keep this tutorial focused, these solutions will not be elaborated upon here. However, readers are encouraged to consult [this script](https://www.mathematik.uni-ulm.de/stochastik/lehre/ss07/seminar_sl/fischer.pdf) by Jasmin Fischer for further explanations.


## 3. How are Support Vector Machines Applied to Classify Data?
<a class="anchor" id="third-section"></a>

Fortunately, you don’t need to calculate any of these equations to use an SVM as a classifier. The **scikit-learn** library provides everything you need!

To get started, first import the SVM module from scikit-learn:

In [ ]:
from sklearn import svm

Next, we create a support vector classifier object:

In [ ]:
clf = svm.SVC(C = 1000, kernel = 'linear')
clf

> _Note:_ The regularization parameter `C` is used to tune the trade-off between the two optimisation objectives discussed earlier. A lower value of `C` allows further maximisation of the margin, even if it leads to incorrect classifications on the training set.
>
> - **Low `C`**: Focuses on minimising $ \frac{1}{2}\|\boldsymbol{w}\|^2$, which increases the margin size, accepting some misclassifications.
> - **High `C`**: Enforces $ y_i (\boldsymbol x_i \cdot \boldsymbol w - b) \geqq 1 \quad \forall i \in \{1,\dots,N\}$, ensuring all training points are correctly classified.
>
> In our case we set the correct classification of the training data as a constraint, which we approximate using a large value of `C`.

> _Note:_ The `kernel` parameter defines the kernel type used by the SVM.
> In this case, we use **`kernel='linear'`** which is suitable for data, where a straight line (or hyperplane) divides the classes.

To compute the **decision function** of our classifier, we need to **fit** it to the input data. 

We use the `help()` function to find out the required format for the input data.

In [ ]:
help(clf.fit)

We find that we need to create a training vector `X` containing all data points: 
* Each **row** of `X` represents a separate **sample**, in our case a flower.
* Each **column** holds the values for a specific **feature**. So far we have chosen the sepal length and sepal width as features.

In [ ]:
X = df[['sepal length (cm)', 'sepal width (cm)']].values
X

Secondly, we need the target vector `y`. This vector contains the class labels corresponding to the training vector. Integers are used to encode the class as follows:

- **_Iris setosa_**: `0`
- **_Iris versicolor_**: `1`

*Note that `scikit-learn` expects labels $y \in \{0, 1\}$ and internally maps them to $y \in \{-1, 1\}$ in the case of SVMs.*


In [ ]:
y = (df['species'].values == 'versicolor').astype(int)
y

We can now fit our training data, which means scikit-learn will handle the optimisation problem presented earlier.

In [ ]:
clf.fit(X, y)

Before using the classifier, let us visualize the support vectors and the separating line:

In [ ]:
# plot separating hyperplane code as from source [1] 
from sklearn.inspection import DecisionBoundaryDisplay

fig4, ax4 = plot_iris(df) # create a plot simular to the one above

# plot the decision function
DecisionBoundaryDisplay.from_estimator(clf, X, plot_method="contour",
    colors="k", levels=[-1, 0, 1], alpha=0.5, linestyles=["--", "-", "--"],
    ax=ax4,
)

# plot support vectors
ax4.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1], s=100,
    linewidth=1, facecolors="none", edgecolors="k",
)

plt.show()

# end of source [1]
# source [1]: https://scikit-learn.org/1.5/auto_examples/svm/plot_separating_hyperplane.html

### Classifying new Samples
We can see that the SVM has indeed found the optimal separating line. Now we can use it to classify new samples:

We measure the **sepal** of a new **Iris flower** and find that it is **6 cm** long and **3.2 cm** wide.


In [ ]:
x = [6.0, 3.2]

The only thing we need to do now is to use the SVM to **classify** our new data point.

In [ ]:
y_predict = clf.predict([x])
y_predict

The classifier predicts a label of **`1`**, which corresponds to **_Iris versicolor_**. Looking at the plot, this indeed seems likely:

In [ ]:
fig5, ax5 = plot_iris(df)
ax5.scatter(x[0], x[1])
ax5.legend(['Iris setosa', 'Iris versicolor', 'Newly measured flower'], loc='upper right')
plt.show()

### Evaluating the SVM Classifier
To verify that we weren't just lucky with our result, we split our dataset into a **training set** and a **test set**. 
* The **training set** is used to train the SVM classifier, so to **learn the relationship between the features and the class labels**.
* The **test set** is used to evaluate the classifier's performance after training. It contains samples that the classifier has never seen before. By comparing the predictions made by the classifier on this test set with the actual labels, we can **assess how well the SVM classifier generalizes to new, unseen data**. 

#### 1. Creating the Train-Test Split
We start by creating the train-test-split:

In [ ]:
from sklearn.model_selection import train_test_split

# split dataset into 75% training and 25% test data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25,
                                                    random_state=5)
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_test: {y_test.shape}")

#### 2. Fitting the SVM Classifier to the Training Data
Next, we fit the SVM classifier to the training data only so that the model learns from the training samples to establish the relationship between the input features and their corresponding class labels:

In [ ]:
clf.fit(X_train, y_train)

#### 3. Predicting Classes for Unseen Samples
Now we can use the SVM to predict the class of previously unseen samples. 

In [ ]:
y_pred = clf.predict(X_test)
print("Predictions of test set samples:")
print(f"  predicated classes: {y_pred}")
print(f"  true classes:       {y_test}")

Fortunately, the SVM classifier successfully classified all the samples correctly, as the predicted classes match the true classes exactly.  Now, using your newly created SVM, you'll be able to classify all _Iris flowers_ in your grandfather's garden in time! 🥳

## 4. Conclusion
<a class="anchor" id="fourth-section"></a>

We've just seen how powerful SVMs can be for data classification.

### Advantages of SVMs
Some advantages of SVMs include:
- As the mathematical formulation is not restricted to two dimensions  (as shown here) but works seamlessly on spaces of arbitrary higher dimension, SVMs **perform well with a big number of features** recorded for each sample $\boldsymbol x$.
- The classification is **transparent**, as it's possible to access the decision function (and even visualise it for up to three features).
- SVMs **generalise well** and are relatively **robust to overfitting**.
- Since the classification depends only on the support vectors, SVMs are very **memory efficient**.

### Disadvantages of SVMs
It is also important to be aware of the disadvantages:
- SVMs are **not a good fit for noisy data** with strongly overlapping classes.
- On **large datasets** SVMs are not very practical.
- **New training data** requires a completely new training of the classifier.
- If the data is [**not linearly separable**](https://en.wikipedia.org/wiki/Linear_separability), the [**kernel trick**](https://en.wikipedia.org/wiki/Kernel_method#Mathematics:_the_kernel_trick) needs to be employed. The non-linear kernel needs to be chosen empirically.

### References:
- [Jamin Fischer - Support Vector Machines (SVM)](https://www.mathematik.uni-ulm.de/stochastik/lehre/ss07/seminar_sl/fischer.pdf)
- [roboticsbiz.com - Pros and cons of Support Vector Machine (SVM)](https://roboticsbiz.com/pros-and-cons-of-support-vector-machine-svm/)
- [Dhiraj K - Top 4 advantages and disadvantages of Support Vector Machine or SVM](https://dhirajkumarblog.medium.com/top-4-advantages-and-disadvantages-of-support-vector-machine-or-svm-a3c06a2b107)
- [Wikipedia - Support vector machine](https://en.wikipedia.org/wiki/Support_vector_machine)
- [Support Vector Machine (SVM) in 2 minutes](https://youtu.be/_YPScrckx28?feature=shared)